# 50.01 Идентифицируемость источников ТТРКГ — rheocardiomonitor_mgtu

Ноутбук раздельно проверяет две постановки: совместную идентифицируемость всех
параметров и оцениваемость одного заранее выбранного целевого параметра.
Полный ранг необходим для первой, но не для второй. Целевой параметр допустим,
если его направление не содержит компоненты в нулевом пространстве оператора.


## Доказательный контракт

Каждая строка якобиана связана с точной записью, конфигурацией, монтажом,
режимом, частотой и хэшем конфигурационного FEM-оператора серии 40. Параметры
имеют единицы и масштабы. Whitening использует принятую проектную ковариацию.

`conditional_design_target_std` описывает условный вычислительный дизайн. Он
не является экспериментальной погрешностью, пока ковариация не включает
межзаписную физиологию, контакт, порядок, прибор и повторяемость. Последовательные
конфигурации разрешают оценку дизайна, но не мгновенную инверсию состояния.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np

from ttrkg_analysis import matrix_diagnostics

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def analysis_key(item):
    fields = ["experiment_id", "subject_id", "record_id", "configuration_id", "montage_id", "side_montage_id", "side_size_mm", "channel_state", "mode"]
    return tuple(item.get(field) for field in fields)


def analyze_design_operator(item):
    names = list(item["parameter_names"])
    units = list(item["parameter_units"])
    scales = np.asarray(item["parameter_scales"], dtype=float)
    jacobian = np.asarray(item["jacobian"], dtype=float)
    covariance = np.asarray(item["measurement_covariance"], dtype=float)
    rows = list(item["measurement_configurations"])
    acceptance = item["acceptance"]
    if jacobian.ndim != 2 or jacobian.shape[1] != len(names) or not np.isfinite(jacobian).all():
        raise ValueError("Размерность или значения якобиана некорректны")
    if len(units) != len(names) or scales.shape != (len(names),) or not np.isfinite(scales).all() or np.any(scales <= 0):
        raise ValueError("Для каждого параметра нужны единица и положительный конечный масштаб")
    if covariance.shape != (jacobian.shape[0], jacobian.shape[0]) or not np.isfinite(covariance).all():
        raise ValueError("Ковариация должна соответствовать строкам якобиана и быть конечной")
    if len(rows) != jacobian.shape[0]:
        raise ValueError("Каждая строка якобиана должна иметь точный ключ измерения")
    row_ids = [row.get("row_id") for row in rows]
    if any(not value for value in row_ids) or len(row_ids) != len(set(row_ids)):
        raise ValueError("row_id должны быть непустыми и уникальными")
    if not np.allclose(covariance, covariance.T, atol=1e-12):
        raise ValueError("Ковариация должна быть симметричной")
    chol = np.linalg.cholesky(covariance)
    scaled = jacobian * scales[None, :]
    whitened = np.linalg.solve(chol, scaled)
    relative_tolerance = float(acceptance["relative_rank_tolerance"])
    diagnostics = matrix_diagnostics(whitened, relative_tolerance)
    target = acceptance["target_parameter"]
    if target not in names:
        raise ValueError("Целевой параметр отсутствует в операторе")
    target_index = names.index(target)
    nullspace = diagnostics["nullspace"]
    target_nullspace_norm = float(np.linalg.norm(nullspace[target_index, :])) if nullspace.size else 0.0
    target_estimable = target_nullspace_norm <= float(acceptance["target_nullspace_tolerance"])
    pseudo_inverse = np.linalg.pinv(whitened, rcond=relative_tolerance)
    covariance_scaled = pseudo_inverse @ pseudo_inverse.T
    target_std = None
    if target_estimable:
        target_std = float(scales[target_index] * np.sqrt(max(covariance_scaled[target_index, target_index], 0.0)))
    joint_identifiable = diagnostics["rank"] == len(names)
    condition_pass = diagnostics["condition"] <= float(acceptance["max_nonzero_subspace_condition"])
    target_design_pass = bool(
        target_estimable and condition_pass and target_std is not None
        and target_std <= float(acceptance["max_conditional_target_std"])
    )
    joint_design_pass = bool(joint_identifiable and condition_pass)
    return {
        "parameter_names": names, "parameter_units": units,
        "rank": diagnostics["rank"], "n_parameters": len(names),
        "singular_values": diagnostics["singular_values"].tolist(),
        "nonzero_subspace_condition": diagnostics["condition"],
        "nullspace": nullspace.tolist(), "joint_identifiable": joint_identifiable,
        "joint_design_pass": joint_design_pass, "target_parameter": target,
        "target_nullspace_projection_norm": target_nullspace_norm,
        "target_estimable": target_estimable,
        "conditional_design_target_std": target_std,
        "target_design_pass": target_design_pass,
        "uncertainty_interpretation": "conditional_design_uncertainty",
    }


def synthetic_tests():
    base = {
        "parameter_names": ["target", "nuisance", "unseen"],
        "parameter_units": ["mL", "1", "1"],
        "parameter_scales": [2.0, 1.0, 1.0],
        "measurement_covariance": [[4.0, 0.0], [0.0, 1.0]],
        "measurement_configurations": [{"row_id": "r1"}, {"row_id": "r2"}],
        "acceptance": {
            "relative_rank_tolerance": 1e-10, "target_parameter": "target",
            "target_nullspace_tolerance": 1e-10,
            "max_nonzero_subspace_condition": 10.0,
            "max_conditional_target_std": 3.0,
        },
    }
    estimable = {**base, "jacobian": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]}
    result = analyze_design_operator(estimable)
    assert not result["joint_identifiable"] and result["target_estimable"] and result["target_design_pass"]
    assert abs(result["conditional_design_target_std"] - 2.0) < 1e-12
    nonestimable = {**base, "acceptance": {**base["acceptance"], "target_parameter": "unseen"}, "jacobian": estimable["jacobian"]}
    result = analyze_design_operator(nonestimable)
    assert not result["target_estimable"] and result["conditional_design_target_std"] is None
    joint = {**base, "measurement_covariance": np.eye(3).tolist(), "measurement_configurations": [{"row_id": "r1"}, {"row_id": "r2"}, {"row_id": "r3"}], "jacobian": np.eye(3).tolist()}
    assert analyze_design_operator(joint)["joint_identifiable"]
    bad = {**estimable, "measurement_covariance": [[1.0, 2.0], [2.0, 1.0]]}
    try:
        analyze_design_operator(bad)
    except np.linalg.LinAlgError:
        pass
    else:
        raise AssertionError("Неположительно определённая ковариация должна отклоняться")

synthetic_tests()
print("50.01 synthetic_self_test: passed")


In [ ]:
if not REAL_MODE:
    print("50.01 real_data_status: blocked_until_traceable_source_operator")
else:
    config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
    config = json.loads(config_path.read_text(encoding="utf-8"))
    contract = config.get("source_identifiability", {})
    if contract.get("status") != "accepted_design_operator" or not contract.get("operator_manifest"):
        raise RuntimeError("Не принят оператор идентифицируемости источников")
    operator_path = Path(contract["operator_manifest"]).expanduser().resolve()
    item = json.loads(operator_path.read_text(encoding="utf-8"))
    if item.get("status") != "accepted_design_operator" or item.get("experiment_id") != "exp02" or item.get("device") != "rheocardiomonitor_mgtu":
        raise RuntimeError("Оператор не принят или относится к другому эксперименту/прибору")
    if not item.get("parameterization_id"):
        raise RuntimeError("Не задана версия параметризации источников")
    if item.get("measurement_covariance_status") != "accepted_design_covariance":
        raise RuntimeError("Не принята проектная ковариация")
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    analysis_dir = derived_root / "exp02" / "analysis"
    fem_path = analysis_dir / "40.04_fem_operator_checked.json"
    fem = json.loads(fem_path.read_text(encoding="utf-8"))
    if fem.get("status") != "accepted_complete_fem_operator_contract" or item.get("source_fem_artifact_sha256") != sha256_file(fem_path):
        raise RuntimeError("Оператор идентифицируемости не связан с принятой версией FEM")
    fem_by_key = {analysis_key(entry): entry for entry in fem.get("operators", [])}
    if not fem_by_key:
        raise RuntimeError("Принятый FEM-артефакт пуст")
    for row in item.get("measurement_configurations", []):
        row_key = analysis_key(row)
        if row_key not in fem_by_key:
            raise RuntimeError(f"Строка оператора не соответствует фактической FEM-конфигурации: {row.get('row_id')}")
        fem_row = fem_by_key[row_key]
        if row.get("fem_operator_manifest_sha256") != fem_row["operator_manifest_sha256"]:
            raise RuntimeError("Строка использует другую версию конфигурационного FEM-оператора")
        if float(row.get("frequency_hz", 0.0)) != float(fem_row["frequency_hz"]):
            raise RuntimeError("Частота строки не совпала с FEM")
    result = analyze_design_operator(item)
    experimental = item.get("experimental_state", {})
    simultaneous = experimental.get("simultaneous_configurations") is True
    repeated = experimental.get("repeatability_status") == "accepted"
    complete_covariance = experimental.get("measurement_covariance_scope") == "complete_including_between_record_physiology_contact_order_and_device"
    if result["target_design_pass"] and simultaneous and repeated and complete_covariance:
        status = "target_design_and_experimental_operator_pass"
    elif result["target_design_pass"]:
        status = "target_design_pass_experimental_inversion_not_authorized"
    else:
        status = "target_design_no_go"
    artifact = {
        "schema_version": 2, "analysis": "50.01_source_identifiability",
        "status": status, "experiment_id": "exp02", "device": "rheocardiomonitor_mgtu",
        "operator_manifest_sha256": sha256_file(operator_path),
        "source_fem_artifact_sha256": sha256_file(fem_path),
        "result": result, "experimental_state": experimental,
        "limitations": [
            "conditional_design_uncertainty_is_not_experimental_accuracy",
            "sequential_configurations_do_not_form_an_instantaneous_inverse_problem",
            "ecg_timing_does_not_add_spatial_rank",
            "source_parameterization_is_a_model_hypothesis",
        ],
    }
    out_path = analysis_dir / "50.01_source_identifiability.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("50.01 real_data_status:", status, out_path)


## Критерий продолжения

В серию 60 допускается только статус
`target_design_and_experimental_operator_pass`. Статус проектного прохождения
без одновременности, повторяемости и полного бюджета не разрешает оценку
функции сердца.
